In [1]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.datasets import wage_panel
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

from pymargins import GComputation, steps  # 0.4.0: Margins -> GComputation

cols = ["lwage", "exper", "educ", "married", "union"]
df = wage_panel.load().reset_index(drop=True)[cols].copy()
print(df.describe().round(2))

         lwage    exper     educ  married    union
count  4360.00  4360.00  4360.00  4360.00  4360.00
mean      1.65     6.51    11.77     0.44     0.24
std       0.53     2.83     1.75     0.50     0.43
min      -3.58     0.00     3.00     0.00     0.00
25%       1.35     4.00    11.00     0.00     0.00
50%       1.67     6.00    12.00     0.00     0.00
75%       1.99     9.00    12.00     1.00     0.00
max       4.05    18.00    16.00     1.00     1.00


In [2]:
rng = np.random.default_rng(7)
p_miss = 1 / (1 + np.exp(-(df["exper"] - df["exper"].mean()) / 2))
miss = rng.uniform(size=len(df)) < 0.30 * p_miss

df_nan = df.copy()
df_nan.loc[miss, "educ"] = np.nan
print(f"missing educ: {int(miss.sum())} rows ({miss.mean():.1%})")

missing educ: 654 rows (15.0%)


In [3]:
imp = IterativeImputer(max_iter=10, random_state=0, sample_posterior=True)


def imputer(frame):
    return pd.DataFrame(imp.fit_transform(frame), columns=frame.columns)

In [4]:
df_init = df_nan.fillna(df_nan.mean(numeric_only=True))
fit = smf.ols("lwage ~ exper + educ + married + union", data=df_init).fit()
m_naive = GComputation(fit, method="bootstrap", B=500, seed=3, scale="identity")
print(m_naive.dydx("educ").summary())

            Graph Result (bootstrap, level=0.95)           
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1009   0.0050     0.1009  0.000    0.0912, 0.1100

n = 4360
plan 326dc98@1 | κ = 0.000


In [5]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # silence the imputer's convergence chatter
    m_mi = GComputation(
        steps.reimpute(steps.input(df_nan), imputer),
        outcome="lwage ~ exper + educ + married + union",
        method="bootstrap",
        B=500,
        seed=3,
        scale="identity",
    )
print(m_mi.dydx("educ").summary())

            Graph Result (bootstrap, level=0.95)           
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1097   0.0054     0.1097  0.000    0.0940, 0.1149

n = 4360
plan 3930732@1 | κ = 0.000


In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_compose = GComputation(
        steps.trim(
            steps.reimpute(steps.input(df_nan), imputer),
            lower=2.0,
            columns=["educ"],
        ),
        outcome="lwage ~ exper + educ + married + union",
        method="bootstrap",
        B=400,
        seed=3,
        scale="identity",
    )
print(m_compose.dydx("educ").summary())

            Graph Result (bootstrap, level=0.95)           
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1070   0.0054     0.1070  0.000    0.0937, 0.1144

n = 4360
plan e86be2f@1 | κ = 0.000


In [7]:
def far_below(frame):
    med = frame["lwage"].median()
    mad = (frame["lwage"] - med).abs().median()
    return frame["lwage"] < med - 5 * mad


print(f"flagged on the full sample: {int(far_below(df).sum())} rows")

df_clean = df[~far_below(df)].reset_index(drop=True)
fit_clean = smf.ols("lwage ~ exper + educ + married + union", data=df_clean).fit()

m_out = GComputation(
    steps.drop_outliers(steps.input(df), far_below),
    outcome=fit_clean,
    method="bootstrap",
    B=500,
    seed=0,
    scale="identity",
)
print(m_out.dydx("educ").summary())

flagged on the full sample: 51 rows


            Graph Result (bootstrap, level=0.95)           
      estimate  std err  statistic  P>|z|  [95% Conf. Int.]
-----------------------------------------------------------
educ    0.1050   0.0037     0.1050  0.000    0.0970, 0.1111

n = 4309
plan 3a7b073@1 | κ = 0.000


In [8]:
try:
    GComputation(
        steps.reimpute(steps.input(df_nan), imputer),
        outcome="lwage ~ exper + educ + married + union",
        method="delta",
        scale="identity",
    )
except ValueError as exc:
    print(exc)

[method_unsupported] method='delta' is not compatible with bootstrap-only transform stages; use method='bootstrap'.


In [9]:
try:
    GComputation(
        steps.drop_outliers(steps.input(df), far_below),
        outcome=fit,
        weights=np.ones(len(df)),
        method="bootstrap",
        scale="identity",
    )
except ValueError as exc:
    print(exc)

[template_mismatch] Template training data fingerprint (42ca513630a3f935...) does not match wiring output fingerprint (5fc54d27bc96c162...).
